# 02 · Train and compare

Three models (logistic regression, random forest, XGBoost), each trained four ways
to handle the imbalance, for the target chosen below:

| Target | One row per | Label | Events |
| --- | --- | --- | --- |
| `loan_default` | loan (`loan_features`) | `defaulted`: rolled over, overdue or lost track | about 1 in 9 |
| `missed_repayment` | member per meeting while a loan runs (`member_months`) | `missed_ahead`: a repayment missed in the next 3 meetings | about 1 in 5 |

| Strategy | What it does to the **training** rows |
| --- | --- |
| `none` | nothing |
| `class_weight` | each event counts as much as all non-events together |
| `kmeans_smote` | k-means groups the rows; synthetic events are added inside groups where events occur |
| `cluster_centroids` | the non-events are shrunk to one real row per k-means group |

Weighting and resampling train as if half the rows were events; the scores are then
moved back to the real share (prior correction), so they still read as risks.

**Tests** (decision D-020): **walk forward**, the main test (each test year is scored
by a model trained only on earlier rows whose outcome was known before that year),
and **members held out** (scored by a model that never saw the member).
**Metrics:** PR AUC (are events ranked first? chance = the share of events), ROC AUC,
and the Brier score (lower is better), with bootstrap 95% intervals. Every run is
logged to MLflow with its plots and model.

> The tables hold member codes and features only: no names, no free text (decision D-016). The loan-default label is **provisional** until the Chama agrees it (D-018): `defaulted` = rolled over while unpaid, overdue (confirmed), or lost track.

In [ ]:
# Setup: runs in Google Colab and on this computer.
# Colab, once: add a GitHub token that can read this repository as the Colab
# secret GITHUB_TOKEN (key icon on the left). This cell mounts Drive, fetches
# the code and installs what Colab lacks. On this computer it uses DRIVE_DIR
# from .env and the `chama` conda environment.
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "m2-features-models"

if IN_COLAB:
    from google.colab import drive, userdata

    REPO = "/content/chama-loan-default"
    OWNER = "muriithi-alvin"
    PLAIN_URL = f"https://github.com/{OWNER}/chama-loan-default.git"

    def git(*args):
        """Run git with the token; show git's message, never the token."""
        token = userdata.get("GITHUB_TOKEN").strip()
        url = PLAIN_URL.replace("https://", f"https://{OWNER}:{token}@")
        result = subprocess.run(["git", *[url if a == "URL" else a for a in args]],
                                capture_output=True, text=True)
        if result.returncode:
            message = (result.stderr or result.stdout).replace(token, "***").strip()
            raise RuntimeError(f"git failed: {message}")

    drive.mount("/content/drive")
    if os.path.exists(REPO):
        git("-C", REPO, "pull", "-q", "URL", BRANCH)
    else:
        git("clone", "-q", "--branch", BRANCH, "URL", REPO)
        git("-C", REPO, "remote", "set-url", "origin", PLAIN_URL)   # no token left behind
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mlflow", "shap", "imbalanced-learn", "python-dotenv"], check=True)
    sys.path.insert(0, f"{REPO}/src")
    os.environ["DRIVE_DIR"] = "/content/drive/MyDrive/chama-loan-default"


import warnings

import pandas as pd

from chama.config import export_dir, load_config, mlflow_dir
from chama.models.data import TARGETS, known_outcomes, load_table, xy
from chama.models.train import target_settings

# the plots are matplotlib Figures built without pyplot: show them as images
%matplotlib inline
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
CONFIG = load_config()
DATA_DIR = export_dir()
print("data:  ", DATA_DIR)
print("MLflow:", mlflow_dir())

In [ ]:
# Which target: "loan_default" (will this loan default?) or
# "missed_repayment" (will a repayment be missed in the next meetings?)
TARGET = "loan_default"

target = TARGETS[TARGET]
SETTINGS = target_settings(CONFIG, TARGET)
DATA = DATA_DIR / f"{target.table}.parquet"
print(TARGET, "| label:", target.label, "| table:", DATA.name)

In [ ]:
from chama.models import train

# All 12 runs take about 8 minutes for loan_default and longer for
# missed_repayment. To try one combination, pass for example
# models=["random_forest"], imbalance=["none", "kmeans_smote"].
summary, scores = train.run(targets=[TARGET], config=CONFIG)
train.summary_table(summary)

## How the risk bands sorted the test rows

Bands are set by share (`band_shares` in `config/pipeline.yaml`, decision D-022): the
top 20% of scores are high, the next 30% medium, the rest low. Each run turns the
shares into score cut-offs from its walk-forward scores. A useful model shows clearly
different event rates per band, with intervals that do not overlap.

In [ ]:
from chama.models.evaluate import band_cutoffs, band_table

best = summary.loc[0, "run"]
best_scores = scores[(TARGET, best)]
cutoffs = band_cutoffs(best_scores["score"], SETTINGS["band_shares"])
print("best walk-forward PR AUC:", best, "| cut-offs:", {k: round(v, 3) for k, v in cutoffs.items()})
band_table(best_scores, cutoffs)

In [ ]:
from IPython.display import display

from chama.models import plots

best_kmeans = summary[summary["imbalance"] == "kmeans_smote"].iloc[0]["run"]
for run_name in dict.fromkeys([best, best_kmeans]):
    display(plots.precision_recall(scores[(TARGET, run_name)], run_name))
    display(plots.calibration(scores[(TARGET, run_name)], run_name))

## Looking at every run in MLflow

On this computer (PowerShell, `chama` environment), with Drive for desktop synced
(use your Drive letter):

```powershell
$env:MLFLOW_ALLOW_FILE_STORE = "true"
mlflow ui --backend-store-uri "file:///H:/My Drive/chama-loan-default/mlruns"
```

Then open http://127.0.0.1:5000, choose the experiment (`chama-loan-default` or
`chama-missed-repayment`), tick runs and press **Compare**. Each `train` call tags
its runs with the same `training_batch`.